<a href="https://colab.research.google.com/github/gurkan-python/istanbul-toplu-tasima-analizi/blob/main/Gurkan_Uysal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Toplu Taşıma Arzı ve Nüfus/İstihdam Mekansal Eşleşme Analizi

In [1]:
!pip install -q pandas openpyxl geopandas shapely pyproj folium matplotlib scipy


In [3]:
import os

os.rename(
    "/content/istanbul_ilceler.geojson.json",
    "/content/istanbul_ilceler.geojson"
)

print("Dosya adı başarıyla düzeltildi.")

Dosya adı başarıyla düzeltildi.


In [6]:
import json
import re

# Önce GeoJSON'daki ilçe isimlerini temizleyelim ve normalize edelim
with open('/content/istanbul_ilceler.geojson', 'r', encoding='utf-8') as f:
    data = json.load(f)

def normalize_tr(text):
    if not text: return ""
    text = text.replace('I', 'ı').replace('İ', 'i').replace('Ş', 'ş').replace('Ç', 'ç').replace('Ğ', 'ğ').replace('Ü', 'ü').replace('Ö', 'ö')
    text = text.lower().strip()
    # İlçe adının sonundaki 'ilçesi' veya 'belediyesi' gibi ekleri temizleyelim
    text = re.sub(r'\s*(ilcesi|belediyesi|ilçe|belediye)$', '', text)
    return text

for feature in data['features']:
    props = feature['properties']
    display_name = props.get('display_name', '')
    address = props.get('address', {})
    ilce_name = address.get('district') or address.get('suburb') or address.get('town') or address.get('city_district')
    if not ilce_name and display_name:
        ilce_name = display_name.split(',')[0].strip()

    # İlçe adının ilk harflerini büyük yapalım ama Türkçe karakter kurallarına uyarak
    if ilce_name:
        ilce_name = ilce_name.replace('i', 'İ').replace('ı', 'I').title()
    props['name'] = ilce_name

with open('/content/istanbul_ilceler.geojson', 'w', encoding='utf-8') as f:
    json.dump(data, f, ensure_ascii=False, indent=2)

# Script dosyasındaki hatayı düzeltelim (hizmet_acigi sütun kontrolü)
with open('/content/istanbul_mekansal_eslesme.py', 'r', encoding='utf-8') as f:
    script_content = f.read()

# Eğer 'valid[valid.hizmet_acigi]' şeklinde filtreleme yapılıyorsa ve hata veriyorsa bunu güvenli hale getirelim
# 'valid.get("hizmet_acigi")' veya '"hizmet_acigi" in valid.columns' kontrolü ekleyelim
old_line = "valid[valid.hizmet_acigi]"
new_line = "valid[valid['hizmet_acigi'] == True]" if "hizmet_acigi" in script_content else old_line

# Alternatif olarak hata veren satırı try-except bloğuna alabiliriz veya doğrudan dataframe kontrolü ekleyebiliriz
script_content = script_content.replace(
    "for _,r in valid[valid.hizmet_acigi].iterrows():",
    "hizmet_acigi_df = valid[valid['hizmet_acigi']] if 'hizmet_acigi' in valid.columns else valid[valid['hizmet_acigi_aday'] == True] if 'hizmet_acigi_aday' in valid.columns else valid\n    for _,r in hizmet_acigi_df.iterrows():"
)

with open('/content/istanbul_mekansal_eslesme.py', 'w', encoding='utf-8') as f:
    f.write(script_content)

print("İsim normalizasyonu ve script düzeltmeleri uygulandı. Analiz tekrar başlatılıyor...")
%run /content/istanbul_mekansal_eslesme.py

İsim normalizasyonu ve script düzeltmeleri uygulandı. Analiz tekrar başlatılıyor...
Excel okunuyor: /content/istanbul_toplu_tasima_birlesik.xlsx
Nüfus yılı: 2025 İlçe sayısı: 39
Geçerli durak: 15386


/usr/local/lib/python3.13/dist-packages/folium/raster_layers.py:130: UserWarning: CartoDB tiles now require an API key. Please provide one to continue using the tiles. You can request the key at https://carto.com/basemaps/apikey/.
  tiles = tiles.build_url(fill_subdomain=False, scale_factor="{r}")  # type: ignore



HİZMET AÇIĞI (ilçe düzeyi adayları):
         ilce  nufus_yogunlugu_km2  onbin_kisi_basina_durak
Gaziosmanpaşa         41049.871276                 4.431484
     Güngören         34954.918220                 4.855876
 Bahçelievler         32597.555513                 4.081368
     Bağcılar         31594.416095                 4.479711
   Bayrampaşa         28639.437360                 4.176161
  Zeytinburnu         24177.154269                 4.828094
     Esenyurt         23362.374337                 4.452613
Eşikler: yoğunluk >= 23075.4 kişi/km²; durak <= 5.59 / 10 bin kişi

Bitti. Çıktılar: /content/analiz_sonuclari
